<a href="https://colab.research.google.com/github/prinshu756/Exoplanet-detection/blob/main/notebooks/02_Light_Curve_Acquisition_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## This is the second notebook

In [1]:
!pip -q install lightkurve astropy astroquery pyarrow tqdm wotan

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.1/261.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 57.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.6.0 which is incompatible.


In [2]:
import os
import gc
import time
import logging
import warnings

from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.notebook import tqdm

import matplotlib.pyplot as plt

import lightkurve as lk

warnings.filterwarnings("ignore")

print("Libraries Imported Successfully")

Libraries Imported Successfully


/usr/local/lib/python3.12/dist-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [5]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
PROJECT_NAME = "AI_Exoplanet_Detection"

ROOT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME

CATALOG_DIR = ROOT_DIR / "data" / "processed"

LIGHTCURVE_DIR = ROOT_DIR / "data" / "raw" / "lightcurves"

LOG_DIR = ROOT_DIR / "logs"

MAX_TARGETS = 100      # None → Entire Catalog

MISSION = "TESS"

AUTHOR = "SPOC"

DOWNLOAD_TIMEOUT = 120

print("Configuration Loaded")

Configuration Loaded


In [7]:
LOG_FILE = LOG_DIR / "notebook_02.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ],
    force=True
)

logger = logging.getLogger("Notebook02")

logger.info("Notebook 02 Started")

2026-07-21 21:29:17,811 | INFO | Notebook 02 Started


In [9]:
catalog = pd.read_parquet(
    CATALOG_DIR / "master_catalog.parquet"
)

if MAX_TARGETS is not None:
    catalog = catalog.head(MAX_TARGETS)

print(f"Targets Loaded : {len(catalog):,}")

catalog.head()

Targets Loaded : 100


,tic_id,planet_name,host_star,ra,dec,orbital_period_days,planet_radius_earth,planet_mass_earth,stellar_teff,stellar_radius,stellar_mass,distance_pc,discovery_year,class_label,class_id,source_catalog
0,158722002,Kepler-1597 b,Kepler-1597,288.163969,42.481012,2.946542,1.06,1.20,6377.0,1.390,1.250,1221.050,2016,Planet,0,NASA Exoplanet Archive
1,121988890,Kepler-687 b,Kepler-687,289.315458,39.153291,20.505870,3.52,12.20,4841.0,0.730,0.770,633.660,2016,Planet,0,NASA Exoplanet Archive
2,350813882,Kepler-1596 b,Kepler-1596,285.557507,49.330087,66.373379,1.90,4.27,5706.0,0.920,0.950,2146.230,2016,Planet,0,NASA Exoplanet Archive
3,138728891,Kepler-692 b,Kepler-692,294.776807,40.153703,21.812935,3.11,9.85,5440.0,0.860,0.900,991.336,2016,Planet,0,NASA Exoplanet Archive
4,121598758,Kepler-150 c,Kepler-150,288.234103,40.520897,7.381998,3.69,13.20,5560.0,0.939,0.956,891.092,2014,Planet,0,NASA Exoplanet Archive


In [10]:
def output_file(tic_id, sector):

    return LIGHTCURVE_DIR / f"TIC_{tic_id}_Sector_{sector}.parquet"


def already_downloaded(tic_id, sector):

    return output_file(tic_id, sector).exists()


def save_lightcurve(df, tic_id, sector):

    df.to_parquet(
        output_file(tic_id, sector),
        index=False
    )

In [11]:
def search_lightcurve(tic_id):

    target = f"TIC {tic_id}"

    result = lk.search_lightcurve(
        target,
        mission=MISSION,
        author=AUTHOR
    )

    return result

In [12]:
def download_lightcurve(search_result):

    if len(search_result) == 0:
        return None

    lc = search_result.download()

    return lc

In [15]:
def lc_to_dataframe(lc, tic_id):

    # Extract metadata safely
    sector = lc.meta.get("SECTOR", np.nan)
    author = lc.meta.get("AUTHOR", "Unknown")
    mission = lc.meta.get("MISSION", "TESS")
    target_name = lc.meta.get("TARGETID", f"TIC {tic_id}")
    cadence = lc.meta.get("EXPOSURE", np.nan)

    df = pd.DataFrame({

        "tic_id": tic_id,

        "sector": sector,

        "mission": mission,

        "author": author,

        "target_name": target_name,

        "cadence_sec": cadence,

        "time": lc.time.value,

        "flux": lc.flux.value,

        "flux_err": lc.flux_err.value,

        "quality": lc.quality

    })

    return df

In [17]:
download_log = []

for _, row in tqdm(catalog.iterrows(), total=len(catalog)):

    tic_id = row["tic_id"]

    try:

        search = search_lightcurve(tic_id)

        if len(search) == 0:

            download_log.append({

                "tic_id": tic_id,

                "status": "Not Found"

            })

            continue

        lc = download_lightcurve(search)

        sector = lc.meta.get("SECTOR", "Unknown")

        if already_downloaded(tic_id, sector):

            download_log.append({

                "tic_id": tic_id,

                "status": "Skipped"

            })

            continue

        df = lc_to_dataframe(lc, tic_id)

        save_lightcurve(df, tic_id, sector)

        download_log.append({

            "tic_id": tic_id,

            "status": "Success",

            "sector": sector,

            "points": len(df)

        })

        del lc
        del df

        gc.collect()

    except Exception as e:

        logger.error(f"{tic_id} : {e}")

        download_log.append({

            "tic_id": tic_id,

            "status": "Failed",

            "error": str(e)

        })

  0%|          | 0/100 [00:00<?, ?it/s]

0% (22/18810) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
2026-07-21 21:34:42,412 | INFO | 0% (22/18810) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
2026-07-21 21:34:42,674 | ERROR | 158722002 : ('Byte-swapped arrays not supported', 'Conversion failed for column quality with type >i4')
0% (41/18597) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
2026-07-21 21:34:42,924 | INFO | 0% (41/18597) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
2026-07-21 21:34:43,114 | ERROR | 121988890 : ('Byte-swapped arrays not supported', 'Conversion failed for column quality with type >i4')
0% (41/18597) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
2026-07-21 21:34:43,466 | INFO | 0% (41/18597) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
2026-07-21 21:34:43,761 | ERROR | 350813882 : ('Byte-sw

In [18]:
report = pd.DataFrame(download_log)

report.to_csv(
    ROOT_DIR / "reports" / "download_report.csv",
    index=False
)

report.head()

,tic_id,status,error
0,158722002,Failed,"('Byte-swapped arrays not supported', 'Convers..."
1,121988890,Failed,"('Byte-swapped arrays not supported', 'Convers..."
2,350813882,Failed,"('Byte-swapped arrays not supported', 'Convers..."
3,138728891,Not Found,NaN
4,121598758,Failed,"('Byte-swapped arrays not supported', 'Convers..."
